In [3]:
import pandas as pd
from google.colab import drive



# 1. Load the file directly from your Google Drive
# (Change 'My Drive/Folder_Name/' if you put it inside a specific folder)
file_path = '/content/drive/MyDrive/csc490/train-00000-of-00001.parquet'
df = pd.read_parquet(file_path)

# 2. Verify it loaded perfectly by checking the column names
print("Successfully loaded columns:", df.columns.tolist())
print(f"Total rows found: {len(df)}")


Successfully loaded columns: ['image', 'text']
Total rows found: 484


In [5]:
import os
import json
import random
import shutil
import asyncio

from bs4 import BeautifulSoup
from PIL import Image
import numpy as np


# ============================================================
# INSTALL / SETUP
# ============================================================

print("Verifying Playwright system binaries...")

os.system("pip install -q playwright beautifulsoup4 pillow numpy")
os.system("playwright install --with-deps chromium")

from playwright.async_api import async_playwright


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = "/content/design2code_mutated_10_6"

IMAGES_DIR = os.path.join(
    OUTPUT_DIR,
    "screenshots"
)

METADATA_DIR = os.path.join(
    OUTPUT_DIR,
    "metadata"
)

os.makedirs(
    IMAGES_DIR,
    exist_ok=True
)

os.makedirs(
    METADATA_DIR,
    exist_ok=True
)


# Fixed screenshot size
VIEWPORT_WIDTH = 1280
VIEWPORT_HEIGHT = 800


# ============================================================
# RANDOM HELPERS
# ============================================================

def random_hex_color():
    """
    Generate a random HEX color.
    """
    return "#{:06X}".format(
        random.randint(0, 0xFFFFFF)
    )


# ============================================================
# CSS HELPER
# ============================================================

def add_style_tag(soup, css):
    """
    Add a new <style> tag to <head>.
    This is generally more reliable than modifying
    random inline styles.
    """

    head = soup.find("head")

    if head is None:

        html = soup.find("html")

        if html is not None:

            head = soup.new_tag("head")

            html.insert(
                0,
                head
            )

        else:

            head = soup.new_tag("head")

            soup.insert(
                0,
                head
            )

    style_tag = soup.new_tag(
        "style"
    )

    style_tag.string = css

    head.append(
        style_tag
    )

    return soup


# ============================================================
# FIXED CANVAS
# ============================================================

def enforce_fixed_canvas(soup):
    """
    Force the HTML document to render inside a fixed
    1280 x 800 canvas.

    This prevents the screenshot size from changing
    when elements are deleted or spacing is modified.
    """

    html = soup.find("html")

    if html is not None:

        existing_style = html.get(
            "style",
            ""
        )

        html["style"] = (
            existing_style
            + f"; width: {VIEWPORT_WIDTH}px !important;"
            + f" height: {VIEWPORT_HEIGHT}px !important;"
            + f" min-width: {VIEWPORT_WIDTH}px !important;"
            + f" min-height: {VIEWPORT_HEIGHT}px !important;"
            + f" max-width: {VIEWPORT_WIDTH}px !important;"
            + f" max-height: {VIEWPORT_HEIGHT}px !important;"
            + "; overflow: hidden !important;"
            + "; box-sizing: border-box !important;"
        )

    body = soup.find("body")

    if body is not None:

        existing_style = body.get(
            "style",
            ""
        )

        body["style"] = (
            existing_style
            + f"; width: {VIEWPORT_WIDTH}px !important;"
            + f" height: {VIEWPORT_HEIGHT}px !important;"
            + f" min-width: {VIEWPORT_WIDTH}px !important;"
            + f" min-height: {VIEWPORT_HEIGHT}px !important;"
            + f" max-width: {VIEWPORT_WIDTH}px !important;"
            + f" max-height: {VIEWPORT_HEIGHT}px !important;"
            + "; overflow: hidden !important;"
            + "; box-sizing: border-box !important;"
        )

    return soup


# ============================================================
# MUTATION 1: TYPOGRAPHY
# ============================================================

def mutate_typography(soup):

    font_family = random.choice([
        "Courier New",
        "Comic Sans MS",
        "Impact",
        "Georgia",
        "Times New Roman",
        "Verdana",
        "Trebuchet MS"
    ])

    font_size = random.choice([
        "8px",
        "10px",
        "14px",
        "24px",
        "32px",
        "40px"
    ])

    line_height = random.choice([
        "0.7",
        "1.8",
        "2.5"
    ])

    css = f"""
    body {{
        font-family: "{font_family}" !important;
        font-size: {font_size} !important;
        line-height: {line_height} !important;
    }}

    body * {{
        font-family: "{font_family}" !important;
    }}

    h1, h2, h3, h4, h5, h6, p, span, a, button {{
        font-size: {font_size} !important;
    }}
    """

    soup = add_style_tag(
        soup,
        css
    )

    return (
        str(soup),
        "regression"
    )


# ============================================================
# MUTATION 2: COLOUR
# ============================================================

def mutate_color(soup):

    background_color = random.choice([
        "#000000",
        "#FF0000",
        "#00FF00",
        "#0000FF",
        "#FFFF00",
        "#FF00FF",
        "#00FFFF",
        "#800080",
        "#FFA500",
        random_hex_color()
    ])

    text_color = random.choice([
        "#FFFFFF",
        "#000000",
        "#FFFF00",
        "#00FFFF",
        "#FF00FF",
        random_hex_color()
    ])

    css = f"""
    body {{
        background-color: {background_color} !important;
        color: {text_color} !important;
    }}

    body * {{
        color: {text_color} !important;
    }}
    """

    soup = add_style_tag(
        soup,
        css
    )

    return (
        str(soup),
        "regression"
    )


# ============================================================
# MUTATION 3: LAYOUT
# ============================================================

def mutate_layout(soup):

    mutation = random.choice([

        "translate_x",

        "translate_y",

        "scale_down",

        "scale_up",

        "move_left",

        "move_down"

    ])

    if mutation == "translate_x":

        css = """
        body > * {
            transform:
                translateX(120px)
                !important;
        }
        """

    elif mutation == "translate_y":

        css = """
        body > * {
            transform:
                translateY(120px)
                !important;
        }
        """

    elif mutation == "scale_down":

        css = """
        body > * {
            transform:
                scale(0.7)
                !important;
            transform-origin:
                top left
                !important;
        }
        """

    elif mutation == "scale_up":

        css = """
        body > * {
            transform:
                scale(1.3)
                !important;
            transform-origin:
                top left
                !important;
        }
        """

    elif mutation == "move_left":

        css = """
        body > * {
            position:
                relative
                !important;
            left:
                150px
                !important;
        }
        """

    else:

        css = """
        body > * {
            position:
                relative
                !important;
            top:
                120px
                !important;
        }
        """

    soup = add_style_tag(
        soup,
        css
    )

    return (
        str(soup),
        "regression"
    )


# ============================================================
# MUTATION 4: SPACING
# ============================================================

def mutate_spacing(soup):

    margin = random.choice([
        "30px",
        "50px",
        "80px",
        "120px",
        "180px"
    ])

    padding = random.choice([
        "20px",
        "40px",
        "80px",
        "120px"
    ])

    css = f"""
    body > * {{
        margin:
            {margin}
            !important;

        padding:
            {padding}
            !important;
    }}
    """

    soup = add_style_tag(
        soup,
        css
    )

    return (
        str(soup),
        "regression"
    )


# ============================================================
# MUTATION 5: DELETION
# ============================================================

def mutate_deletion(soup):

    # Prefer visually meaningful elements
    elements = soup.find_all([
        "header",
        "nav",
        "main",
        "section",
        "article",
        "footer",
        "h1",
        "h2",
        "h3",
        "img",
        "button"
    ])

    if elements:

        # Randomly delete exactly ONE object
        target = random.choice(
            elements
        )

        target.decompose()

    else:

        # Fallback: choose from generic containers
        elements = soup.find_all([
            "div",
            "p"
        ])

        if elements:

            target = random.choice(
                elements
            )

            target.decompose()

    return (
        str(soup),
        "regression"
    )


# ============================================================
# MUTATION 6: SIZING
# ============================================================

def mutate_sizing(soup):

    width = random.choice([
        "10%",
        "25%",
        "40%",
        "60%",
        "75%"
    ])

    css = f"""
    body > * {{
        width:
            {width}
            !important;

        max-width:
            {width}
            !important;

        box-sizing:
            border-box
            !important;
    }}
    """

    soup = add_style_tag(
        soup,
        css
    )

    return (
        str(soup),
        "regression"
    )


# ============================================================
# MUTATION 7: VISIBILITY
# ============================================================

def mutate_visibility(soup):

    opacity = random.choice([
        "0.05",
        "0.10",
        "0.20",
        "0.30"
    ])

    blur = random.choice([
        "3px",
        "5px",
        "8px",
        "12px"
    ])

    css = f"""
    body > * {{
        opacity:
            {opacity}
            !important;

        filter:
            blur({blur})
            !important;
    }}
    """

    soup = add_style_tag(
        soup,
        css
    )

    return (
        str(soup),
        "regression"
    )


# ============================================================
# MUTATION MAP
# ============================================================

MUTATION_MAP = {

    "typography":
        mutate_typography,

    "colour":
        mutate_color,

    "layout":
        mutate_layout,

    "spacing":
        mutate_spacing,

    "deletion":
        mutate_deletion,

    "sizing":
        mutate_sizing,

    "visibility":
        mutate_visibility
}


# ============================================================
# PIXEL DIFFERENCE
# ============================================================

def calculate_pixel_difference(
    image_path_1,
    image_path_2
):
    """
    Calculate mean absolute pixel difference.

    0 means the two images are identical.
    Larger values mean greater visual difference.
    """

    try:

        img1 = Image.open(
            image_path_1
        ).convert("RGB")

        img2 = Image.open(
            image_path_2
        ).convert("RGB")

        # Ensure same size
        if img1.size != img2.size:
            return None

        arr1 = np.array(
            img1
        ).astype(
            np.float32
        )

        arr2 = np.array(
            img2
        ).astype(
            np.float32
        )

        difference = np.mean(
            np.abs(
                arr1 - arr2
            )
        )

        return round(
            float(difference),
            4
        )

    except Exception:

        return None


# ============================================================
# PIPELINE
# ============================================================

async def run_pipeline(dataframe):

    print(
        f"Starting processing directly from loaded dataframe "
        f"with {len(dataframe)} rows..."
    )

    metadata_records = []

    async with async_playwright() as p:

        # ----------------------------------------------------
        # Launch browser
        # ----------------------------------------------------

        browser = await p.chromium.launch(
            headless=True
        )

        # ----------------------------------------------------
        # Create page with fixed viewport
        # ----------------------------------------------------

        page = await browser.new_page(
            viewport={
                "width":
                    VIEWPORT_WIDTH,

                "height":
                    VIEWPORT_HEIGHT
            }
        )

        # ----------------------------------------------------
        # Default timeout
        # ----------------------------------------------------

        page.set_default_timeout(
            10000
        )

        # ----------------------------------------------------
        # Process each original
        # ----------------------------------------------------

        for index, row in dataframe.iterrows():

            parent_id = (
                f"d2c_{index:03d}"
            )

            raw_html = row.get(
                "text"
            )

            if not raw_html:

                print(
                    f"Skipping {parent_id}: "
                    f"empty HTML."
                )

                continue

            print(
                f"Mutating item "
                f"[{index + 1}/{len(dataframe)}] "
                f"(ID: {parent_id})..."
            )

            # =================================================
            # IMPORTANT:
            #
            # Every mutation starts from ORIGINAL HTML.
            #
            # They do NOT accumulate.
            # =================================================

            for mutation_name, mutation_func in MUTATION_MAP.items():

                mutation_id = (
                    f"{parent_id}_"
                    f"{mutation_name}"
                )

                try:

                    # -----------------------------------------
                    # Fresh HTML for every mutation
                    # -----------------------------------------

                    soup_instance = BeautifulSoup(
                        raw_html,
                        "html.parser"
                    )

                    # -----------------------------------------
                    # Apply mutation
                    # -----------------------------------------

                    mutated_html, label = (
                        mutation_func(
                            soup_instance
                        )
                    )

                    # -----------------------------------------
                    # Re-parse after mutation
                    # -----------------------------------------

                    mutated_soup = BeautifulSoup(
                        mutated_html,
                        "html.parser"
                    )

                    # -----------------------------------------
                    # Force fixed canvas
                    # -----------------------------------------

                    mutated_soup = (
                        enforce_fixed_canvas(
                            mutated_soup
                        )
                    )

                    mutated_html = str(
                        mutated_soup
                    )

                    # -----------------------------------------
                    # Render
                    #
                    # IMPORTANT:
                    # Use domcontentloaded instead of load.
                    # This avoids hanging on external resources.
                    # -----------------------------------------

                    await page.set_content(
                        mutated_html,
                        wait_until="domcontentloaded",
                        timeout=10000
                    )

                    # -----------------------------------------
                    # Give browser a short rendering period
                    # -----------------------------------------

                    await page.wait_for_timeout(
                        500
                    )

                    # -----------------------------------------
                    # Screenshot
                    #
                    # full_page=False ensures fixed size.
                    # -----------------------------------------

                    screenshot_filename = (
                        f"{mutation_id}.png"
                    )

                    screenshot_path = os.path.join(
                        IMAGES_DIR,
                        screenshot_filename
                    )

                    await page.screenshot(
                        path=screenshot_path,
                        full_page=False
                    )

                    # -----------------------------------------
                    # Verify image dimensions
                    # -----------------------------------------

                    generated_image = Image.open(
                        screenshot_path
                    )

                    image_width, image_height = (
                        generated_image.size
                    )

                    # -----------------------------------------
                    # Try to locate original screenshot
                    #
                    # This is optional. If dataframe has an
                    # image path, we can calculate difference.
                    # -----------------------------------------

                    original_image_path = None

                    possible_columns = [
                        "image_path",
                        "screenshot_path",
                        "original_image",
                        "image"
                    ]

                    for column in possible_columns:

                        if column in dataframe.columns:

                            candidate = row.get(
                                column
                            )

                            if (
                                candidate
                                and isinstance(
                                    candidate,
                                    str
                                )
                                and os.path.exists(
                                    candidate
                                )
                            ):

                                original_image_path = (
                                    candidate
                                )

                                break

                    # -----------------------------------------
                    # Pixel difference
                    # -----------------------------------------

                    pixel_difference = None

                    if original_image_path:

                        pixel_difference = (
                            calculate_pixel_difference(
                                original_image_path,
                                screenshot_path
                            )
                        )

                    # -----------------------------------------
                    # Metadata
                    # -----------------------------------------

                    record = {

                        "mutation_id":
                            mutation_id,

                        "parent_id":
                            parent_id,

                        "mutation_type":
                            mutation_name,

                        # EVERYTHING IS REGRESSION
                        "label":
                            "regression",

                        "screenshot_path":
                            screenshot_path,

                        "mutated_html":
                            mutated_html,

                        "image_width":
                            image_width,

                        "image_height":
                            image_height,

                        "pixel_difference":
                            pixel_difference
                    }

                    metadata_records.append(
                        record
                    )

                    # -----------------------------------------
                    # Save individual JSON
                    # -----------------------------------------

                    metadata_path = os.path.join(
                        METADATA_DIR,
                        f"{mutation_id}.json"
                    )

                    with open(
                        metadata_path,
                        "w"
                    ) as f:

                        json.dump(
                            record,
                            f,
                            indent=4
                        )

                    # -----------------------------------------
                    # Progress information
                    # -----------------------------------------

                    print(
                        f"  ✓ {mutation_name}"
                        f" | size={image_width}x{image_height}"
                        + (
                            f" | pixel_diff="
                            f"{pixel_difference}"
                            if pixel_difference is not None
                            else ""
                        )
                    )

                except Exception as e:

                    print(
                        f"  ✗ Warning: "
                        f"Render failed on "
                        f"{mutation_id}: "
                        f"{e}"
                    )

                    continue

        # ----------------------------------------------------
        # Close browser
        # ----------------------------------------------------

        await browser.close()

    # ========================================================
    # MASTER DATASET
    # ========================================================

    master_json_path = os.path.join(
        OUTPUT_DIR,
        "master_dataset.json"
    )

    with open(
        master_json_path,
        "w"
    ) as f:

        json.dump(
            metadata_records,
            f,
            indent=4
        )

    # ========================================================
    # SUMMARY
    # ========================================================

    expected_count = (
        len(dataframe)
        * len(MUTATION_MAP)
    )

    actual_count = len(
        metadata_records
    )

    print("\n")
    print("=" * 60)
    print("PROCESSING COMPLETE")
    print("=" * 60)

    print(
        f"Original HTML rows: "
        f"{len(dataframe)}"
    )

    print(
        f"Mutation types per original: "
        f"{len(MUTATION_MAP)}"
    )

    print(
        f"Expected images: "
        f"{expected_count}"
    )

    print(
        f"Generated images: "
        f"{actual_count}"
    )

    print(
        f"Failed images: "
        f"{expected_count - actual_count}"
    )

    print(
        f"Image size: "
        f"{VIEWPORT_WIDTH} x "
        f"{VIEWPORT_HEIGHT}"
    )

    print(
        f"Label: regression"
    )

    # ========================================================
    # ZIP
    # ========================================================

    if actual_count > 0:

        print("\nCreating ZIP archive...")

        shutil.make_archive(
            base_name="/content/design2code_mutated_10_6",
            format="zip",
            root_dir=OUTPUT_DIR
        )

        zip_path = (
            "/content/design2code_mutated_10_6.zip"
        )

        print(
            f"ZIP created: {zip_path}"
        )

        # ====================================================
        # DOWNLOAD
        # ====================================================

        try:

            from google.colab import files

            print(
                "Downloading ZIP package..."
            )

            files.download(
                zip_path
            )

        except Exception as e:

            print(
                f"Automatic download failed: {e}"
            )

            print(
                f"ZIP is still available at: "
                f"{zip_path}"
            )

    else:

        print(
            "ERROR: No mutations were generated."
        )


# ============================================================
# RUN PIPELINE
# ============================================================

await run_pipeline(df)

Verifying Playwright system binaries...
Starting processing directly from loaded dataframe with 484 rows...
Mutating item [1/484] (ID: d2c_000)...
  ✓ typography | size=1280x800
  ✓ colour | size=1280x800
  ✓ layout | size=1280x800
  ✓ spacing | size=1280x800
  ✓ deletion | size=1280x800
  ✓ sizing | size=1280x800
  ✓ visibility | size=1280x800
Mutating item [2/484] (ID: d2c_001)...
  ✓ typography | size=1280x800
  ✓ colour | size=1280x800
  ✓ layout | size=1280x800
  ✓ spacing | size=1280x800
  ✓ deletion | size=1280x800
  ✓ sizing | size=1280x800
  ✓ visibility | size=1280x800
Mutating item [3/484] (ID: d2c_002)...
  ✓ typography | size=1280x800
  ✓ colour | size=1280x800
  ✓ layout | size=1280x800
  ✓ spacing | size=1280x800
  ✓ deletion | size=1280x800
  ✓ sizing | size=1280x800
  ✓ visibility | size=1280x800
Mutating item [4/484] (ID: d2c_003)...
  ✓ typography | size=1280x800
  ✓ colour | size=1280x800
  ✓ layout | size=1280x800
  ✓ spacing | size=1280x800
  ✓ deletion | size=128

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>